<a href="https://colab.research.google.com/github/Madhav-Sharma91/RFL-python-internship/blob/main/day28project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install streamlit
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

st.set_page_config(
    page_title="Stock Market Portfolio Analyzer",
    page_icon="📈",
    layout="wide"
)

st.title("📈 Stock Market Portfolio Analyzer")
st.caption("Analyze portfolio performance, sector allocation, daily returns, and moving-average trends.")


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def calculate_metrics(df):
    """Calculate stock-level portfolio metrics."""

    latest = (
        df.sort_values("Date")
        .groupby("Ticker")
        .tail(1)
        .copy()
    )

    latest["Investment"] = latest["Quantity"] * latest["Buy_Price"]
    latest["Current_Value"] = latest["Quantity"] * latest["Close"]
    latest["Profit_Loss"] = latest["Current_Value"] - latest["Investment"]

    latest["Return_%"] = np.where(
        latest["Investment"] != 0,
        latest["Profit_Loss"] / latest["Investment"] * 100,
        0
    )

    return latest


def calculate_daily_returns(df):
    """Calculate daily portfolio returns."""

    daily = (
        df.groupby("Date")["Close"]
        .mean()
        .sort_index()
        .to_frame("Portfolio_Value")
    )

    daily["Daily_Return_%"] = daily["Portfolio_Value"].pct_change() * 100

    return daily.reset_index()


def predict_trend(df, window):
    """Predict next-day trend using moving average."""

    data = df.sort_values("Date").copy()

    data["Moving_Average"] = (
        data["Close"]
        .rolling(window=window)
        .mean()
    )

    latest = data.iloc[-1]

    if pd.isna(latest["Moving_Average"]):
        return "Insufficient Data", data

    if latest["Close"] > latest["Moving_Average"]:
        trend = "📈 Bullish"
    elif latest["Close"] < latest["Moving_Average"]:
        trend = "📉 Bearish"
    else:
        trend = "➡️ Neutral"

    return trend, data


# ---------------------------------------------------------
# Sidebar
# ---------------------------------------------------------

st.sidebar.header("⚙️ Settings")

uploaded_file = st.sidebar.file_uploader(
    "Upload portfolio CSV",
    type=["csv"]
)


# ---------------------------------------------------------
# Demo data
# ---------------------------------------------------------

if uploaded_file is None:

    st.info(
        "Upload a CSV from the sidebar. A demo portfolio is displayed below "
        "so you can see how the dashboard works."
    )

    dates = pd.date_range(
        start="2026-01-01",
        periods=120,
        freq="B"
    )

    np.random.seed(42)

    demo_stocks = {
        "AAPL": ("Technology", 220),
        "MSFT": ("Technology", 400),
        "JPM": ("Financials", 270),
        "AMZN": ("Consumer", 180),
        "XOM": ("Energy", 110),
    }

    rows = []

    for ticker, (sector, buy_price) in demo_stocks.items():

        prices = buy_price * np.cumprod(
            1 + np.random.normal(0.0005, 0.018, len(dates))
        )

        quantity = np.random.randint(5, 20)

        for date, price in zip(dates, prices):
            rows.append({
                "Date": date,
                "Ticker": ticker,
                "Close": round(price, 2),
                "Quantity": quantity,
                "Buy_Price": buy_price,
                "Sector": sector
            })

    df = pd.DataFrame(rows)

else:

    df = pd.read_csv(uploaded_file)


# ---------------------------------------------------------
# Normalize columns
# ---------------------------------------------------------

st.subheader("📄 Data Preview")

st.dataframe(df.head(10), use_container_width=True)


st.sidebar.subheader("Column Mapping")

columns = df.columns.tolist()

# Helper function to get the index of a column safely
def get_col_index(col_name, col_list):
    try:
        return col_list.index(col_name)
    except ValueError:
        return 0 # Default to the first column if not found

date_col = st.sidebar.selectbox("Date column", columns, index=get_col_index("Date", columns))
ticker_col = st.sidebar.selectbox("Ticker column", columns, index=get_col_index("Ticker", columns))
close_col = st.sidebar.selectbox("Close/Price column", columns, index=get_col_index("Close", columns))
quantity_col = st.sidebar.selectbox("Quantity column", columns, index=get_col_index("Quantity", columns))
buy_price_col = st.sidebar.selectbox("Buy price column", columns, index=get_col_index("Buy_Price", columns))
sector_col = st.sidebar.selectbox("Sector column", columns, index=get_col_index("Sector", columns))


df = df.rename(columns={
    date_col: "Date",
    ticker_col: "Ticker",
    close_col: "Close",
    quantity_col: "Quantity",
    buy_price_col: "Buy_Price",
    sector_col: "Sector"
})

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_columns = [
    "Close",
    "Quantity",
    "Buy_Price"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=[
        "Date",
        "Ticker",
        "Close",
        "Quantity",
        "Buy_Price",
        "Sector"
    ]
)

df = df.sort_values("Date")


# ---------------------------------------------------------
# Stock metrics
# ---------------------------------------------------------

metrics = calculate_metrics(df)

total_investment = metrics["Investment"].sum()
current_value = metrics["Current_Value"].sum()
total_profit_loss = metrics["Profit_Loss"].sum()

overall_return = (
    total_profit_loss / total_investment * 100
    if total_investment != 0
    else 0
)


# ---------------------------------------------------------
# KPI Dashboard
# ---------------------------------------------------------

st.subheader("📊 Portfolio Overview")

col1, col2, col3, col4 = st.columns(4)

col1.metric(
    "Total Investment",
    f"₹{total_investment:,.2f}"
)

col2.metric(
    "Current Value",
    f"₹{current_value:,.2f}"
)

col3.metric(
    "Profit / Loss",
    f"₹{total_profit_loss:,.2f}",
    delta=f"{overall_return:.2f}%"
)

col4.metric(
    "Stocks",
    metrics["Ticker"].nunique()
)


# ---------------------------------------------------------
# Best / Worst Stocks
# ---------------------------------------------------------

st.subheader("🏆 Best & Worst Performing Stocks")

best_stock = metrics.loc[
    metrics["Return_%"].idxmax()
]

worst_stock = metrics.loc[
    metrics["Return_%"].idxmin()
]

col1, col2 = st.columns(2)

with col1:
    st.success(
        f"""
        **Best Performer:** {best_stock["Ticker"]}

        Return: **{best_stock["Return_%"]:.2f}%**

        Profit/Loss: **₹{best_stock["Profit_Loss"]:,.2f}**
        """
    )

with col2:
    st.error(
        f"""
        **Worst Performer:** {worst_stock["Ticker"]}

        Return: **{worst_stock["Return_%"]:.2f}%**

        Profit/Loss: **₹{worst_stock["Profit_Loss"]:,.2f}**
        """
    )


# ---------------------------------------------------------
# Stock performance table
# ---------------------------------------------------------

st.subheader("📋 Stock Performance")

display_metrics = metrics[
    [
        "Ticker",
        "Sector",
        "Quantity",
        "Buy_Price",
        "Close",
        "Investment",
        "Current_Value",
        "Profit_Loss",
        "Return_%"
    ]
].copy()

display_metrics.columns = [
    "Ticker",
    "Sector",
    "Quantity",
    "Buy Price",
    "Current Price",
    "Investment",
    "Current Value",
    "Profit/Loss",
    "Return %"
]

st.dataframe(
    display_metrics.style.format({
        "Buy Price": "₹{:.2f}",
        "Current Price": "₹{:.2f}",
        "Investment": "₹{:,.2f}",
        "Current Value": "₹{:,.2f}",
        "Profit/Loss": "₹{:,.2f}",
        "Return %": "{:.2f}%"
    }),
    use_container_width=True
)


# ---------------------------------------------------------
# Portfolio Growth Chart
# ---------------------------------------------------------

st.subheader("📈 Portfolio Growth")

portfolio = (
    df.groupby("Date")
    .apply(
        lambda x: (x["Close"] * x["Quantity"]).sum()
    )
    .reset_index(name="Portfolio_Value")
)

fig_growth = px.line(
    portfolio,
    x="Date",
    y="Portfolio_Value",
    title="Portfolio Value Over Time",
    labels={
        "Portfolio_Value": "Portfolio Value",
        "Date": "Date"
    }
)

fig_growth.update_layout(
    hovermode="x unified"
)

st.plotly_chart(
    fig_growth,
    use_container_width=True
)


# ---------------------------------------------------------
# Sector-wise Investment
# ---------------------------------------------------------

st.subheader("📊 Sector-wise Investment")

sector_data = (
    metrics
    .groupby("Sector")["Investment"]
    .sum()
    .reset_index()
)

fig_sector = px.pie(
    sector_data,
    names="Sector",
    values="Investment",
    hole=0.4,
    title="Portfolio Investment by Sector"
)

st.plotly_chart(
    fig_sector,
    use_container_width=True
)


# ---------------------------------------------------------
# Daily Return Analysis
# ---------------------------------------------------------

st.subheader("📉 Daily Return Analysis")

daily_returns = calculate_daily_returns(df)

fig_returns = px.line(
    daily_returns,
    x="Date",
    y="Daily_Return_%",
    title="Daily Portfolio Returns",
    labels={
        "Daily_Return_%": "Daily Return (%)",
        "Date": "Date"
    }
)

fig_returns.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

fig_returns.update_layout(
    hovermode="x unified"
)

st.plotly_chart(
    fig_returns,
    use_container_width=True
)


# ---------------------------------------------------------
# Return statistics
# ---------------------------------------------------------

st.subheader("📐 Return Statistics")

mean_daily_return = daily_returns["Daily_Return_%"].mean()
volatility = daily_returns["Daily_Return_%"].std()

col1, col2, col3 = st.columns(3)

col1.metric(
    "Average Daily Return",
    f"{mean_daily_return:.2f}%"
)

col2.metric(
    "Daily Volatility",
    f"{volatility:.2f}%"
)

col3.metric(
    "Overall Return",
    f"{overall_return:.2f}%"
)


# ---------------------------------------------------------
# Moving Average Prediction
# ---------------------------------------------------------

st.subheader("⭐ Next-Day Trend Prediction")

tickers = sorted(df["Ticker"].unique())

selected_ticker = st.selectbox(
    "Select stock",
    tickers
)

ma_window = st.slider(
    "Moving Average Window",
    min_value=3,
    max_value=50,
    value=20
)

stock_df = (
    df[df["Ticker"] == selected_ticker]
    .sort_values("Date")
    .copy()
)

trend, ma_data = predict_trend(
    stock_df,
    ma_window
)

latest_price = stock_df["Close"].iloc[-1]
latest_ma = ma_data["Moving_Average"].iloc[-1]

col1, col2, col3 = st.columns(3)

col1.metric(
    "Latest Price",
    f"₹{latest_price:,.2f}"
)

col2.metric(
    f"{ma_window}-Day Moving Average",
    f"₹{latest_ma:,.2f}"
    if not pd.isna(latest_ma)
    else "N/A"
)

col3.metric(
    "Predicted Trend",
    trend
)


# Moving-average chart

fig_ma = px.line(
    ma_data,
    x="Date",
    y=["Close", "Moving_Average"],
    title=f"{selected_ticker} Price vs {ma_window}-Day Moving Average",
    labels={
        "value": "Price",
        "variable": "Metric"
    }
)

st.plotly_chart(
    fig_ma,
    use_container_width=True
)


# ---------------------------------------------------------
# Download results
# ---------------------------------------------------------

st.subheader("⬇️ Export Analysis")

csv = display_metrics.to_csv(index=False)

st.download_button(
    label="Download Stock Analysis CSV",
    data=csv,
    file_name="portfolio_analysis.csv",
    mime="text/csv"
)


st.caption(
    "⚠️ Moving-average trend prediction is a simple technical indicator, "
    "not financial advice or a guarantee of future returns."
)


2026-09-04 05:54:34.156 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.157 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.159 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.159 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.161 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:54:34.164 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()